In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

### Load Data

In [6]:
data = pd.read_csv('hw_6_data.csv')
data['image_number'] = data['image_number'].astype(str)

### Augment the Images

In [7]:
# Set up the augmenter
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Create a directory to save augmented images
augmented_dir = '/Users/greysonnewton/Library/CloudStorage/OneDrive-UniversityOfHouston/FALL_2025/Big Data Analytics (PETR)/big-data-analytics/big-data-analytics/homeworks/hw6/images/augmented'
raw_dir = '/Users/greysonnewton/Library/CloudStorage/OneDrive-UniversityOfHouston/FALL_2025/Big Data Analytics (PETR)/big-data-analytics/big-data-analytics/homeworks/hw6/images/raw'
os.makedirs(augmented_dir, exist_ok=True)



# Unzip all images
# def unzip(zip_file, destination):
#     with zipfile.ZipFile(zip_file, 'r') as zf:
#         zf.extractall(destination)
# unzip(zip_file='1000images.zip', destination='./')


# Augment all images
for idx, row in data.iterrows():
    img_number = row['image_number']
    img_path = os.path.join(raw_dir, f'{img_number}.JPG')
    img = load_img(img_path)
    x = img_to_array(img)
    x = x.reshape((1,) + x.shape)

    # Augment 10 times
    for i, batch in enumerate(datagen.flow(x, batch_size=1, save_to_dir=augmented_dir, save_prefix=f"{idx}", save_format='jpeg')):
        if i >= 9:  # 10 images total
            break

### Create New Labels for Augmented Images

In [9]:
# Create new dataframe for augmented images
augmented_filenames = os.listdir(augmented_dir)

# Assume filename is like '0_0.jpeg', '0_1.jpeg', etc.
new_rows = []
for filename in augmented_filenames:
    orig_idx = int(filename.split('_')[0])  # recover original index
    new_rows.append({
        'image_filename': os.path.join(augmented_dir, filename),
        'E (GPa)': data.loc[orig_idx, 'E (GPa)'],
        'Penetration_depth (nm)': data.loc[orig_idx,'Penetration_depth (nm)'],
        'Load (mN)':data.loc[orig_idx, 'Load (mN)'],

    })

augmented_data = pd.DataFrame(new_rows)

# Shuffle the data
augmented_data = augmented_data.sample(frac=1, random_state=42).reset_index(drop=True)
augmented_data

,image_filename,elastic_modulus
0,/Users/greysonnewton/Library/CloudStorage/OneD...,78.90
1,/Users/greysonnewton/Library/CloudStorage/OneD...,75.18
2,/Users/greysonnewton/Library/CloudStorage/OneD...,66.30
3,/Users/greysonnewton/Library/CloudStorage/OneD...,76.98
4,/Users/greysonnewton/Library/CloudStorage/OneD...,74.98
...,...,...
9990,/Users/greysonnewton/Library/CloudStorage/OneD...,76.66
9991,/Users/greysonnewton/Library/CloudStorage/OneD...,68.30
9992,/Users/greysonnewton/Library/CloudStorage/OneD...,76.73
9993,/Users/greysonnewton/Library/CloudStorage/OneD...,73.46


### Load Images and Other Features

In [ ]:
# # Load images into numpy arrays
# def load_images(filenames):
#     images = []
#     for filename in filenames:
#         img = load_img(filename, target_size=(224, 224))
#         img_array = img_to_array(img)
#         images.append(img_array)
#     return np.array(images)
#
# # Load all the images
# X_img = load_images(augmented_data['image_filename'])
#
# # Normalize images
# X_img = X_img / 255.0
#
# # Load Penetration depth and Load (as tabular data)
# X_tabular = augmented_data[['Penetration_depth (nm)', 'Load (mN)']].values
#
# # Target variable: Elastic Modulus
# y = augmented_data['E (GPa)'].values

In [5]:
# import os
# import cv2
#
# from .image_augmentor import ImageAugmentor
#
#
# def augment_images():
#     images = {}
#     for filename in os.listdir('raw'):
#         if filename.endswith('.jpg'):
#             raw_image = cv2.imread('./raw/' + filename)
#             augmentor = ImageAugmentor(raw_image)
#             augmentor.crop(mode='center',crop_factor=0.75)
#             augmentor.random_translation()
#             augmentor.random_brightness(max_delta=75)
#             images[filename] = ImageAugmentor.image
#             ImageAugmentor.reset()
#     return images